# TactiFoot Vision: from dataset to tactical map

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RedHot099/TactiFootVision/blob/master/notebooks/00_overview.ipynb)

`tactifoot_vision` turns football video into tactical data: who is where on
the pitch, in which team, frame by frame. This notebook runs the whole
workflow once on open data, one short step per stage: it downloads a demo
match, trains a player detector on it for a few minutes, tracks the players
through a clip and draws them on the pitch. It needs no local data or
weights, so it runs the same from a checkout and on Google Colab.

**On Colab**, pick a GPU first (Runtime, Change runtime type, T4 GPU) and add
your Hugging Face token as a secret named `HF_TOKEN` (the key icon on the
left) with notebook access switched on; step 1 says why.

Each stage has its own notebook with the details. Those still read local data
and run from a checkout only:

| stage | module | notebook |
|---|---|---|
| Data preparation | `tv.data` | [01_data](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/01_data.ipynb) |
| Data augmentation | `tv.augment` | [02_augmentation](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/02_augmentation.ipynb) |
| Training and evaluation | `tv.models`, `tv.evaluation` | [03_training_and_evaluation](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/03_training_and_evaluation.ipynb) |
| Inference | `tv.tracking`, `tv.pitch`, `tv.teams`, `tv.pipeline` | [04_inference](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/04_inference.ipynb) |
| Visualisation | `tv.viz` | [05_visualisation](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/05_visualisation.ipynb) |

Everything is plain Python: build the objects you want and call them. The
same steps exist as a command line (`tactifoot run`, `tactifoot train`,
`tactifoot evaluate`), described under "Run files and the CLI" in
[docs/ARCHITECTURE.md](https://github.com/RedHot099/TactiFootVision/blob/master/docs/ARCHITECTURE.md).

In [ ]:
import importlib.util

if importlib.util.find_spec("tactifoot_vision") is None:  # e.g. a fresh Colab runtime
    %pip install -q tactifoot-vision

import os
from pathlib import Path

import tactifoot_vision as tv

# The repository root when run from a checkout, else the working directory
# (/content on Colab); kept relative, so printed paths are the same everywhere.
_here = [Path.cwd(), *Path.cwd().parents]
_root = next((p for p in _here if (p / "pyproject.toml").is_file()), Path.cwd())
ROOT = Path(os.path.relpath(_root))
DATA = ROOT / "data"
OUT = ROOT / "outputs" / "notebooks" / "00_overview"

tv.setup_logging("WARNING")
# Most figures are video frames: JPEG keeps the saved notebook small.
%config InlineBackend.figure_formats = ["jpeg"]
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pil_kwargs": {"quality": 85}}

## 1. Data

`tv.data.download_demo` fetches one match of
[SoccerTrack v2](https://huggingface.co/datasets/atomscott/soccertrack-v2)
(about 300 MB, CC BY 4.0) and lays it out for the package: a detection
dataset of frames from the first minutes and a 30-second clip with tracking
truth from the end. The dataset is gated: accept its conditions once on its
page, then the call finds your token in the `HF_TOKEN` Colab secret, the
environment variable of that name or a saved `hf auth login`. A second call
does nothing.

The footage is one static camera that sees the whole pitch through a fisheye
lens, 4096x1080 pixels, so a player is about 25x60 pixels. The labels are
the 22 players; this data has no ball, referee or goalkeeper boxes.

In [ ]:
demo = tv.data.download_demo(DATA / "demo")
dataset = tv.load_dataset(demo.detect_yaml)
print(dataset)
tv.viz.show_samples(dataset, "train", n=2, cols=1)

## 2. Augmentation

Transforms move boxes and keypoints with the pixels. `augment_dataset` adds
augmented copies of the training images; validation images stay real. It
never overwrites earlier output, so a re-run starts from an empty folder.

In [ ]:
import shutil

recipe = tv.augment.Compose(
    [
        tv.augment.HorizontalFlip(),
        tv.augment.RandomAffine(degrees=3, translate=0.05),
        tv.augment.ColorJitter(),
    ]
)
small = dataset.subset({"train": 40, "valid": 10}, seed=0)
shutil.rmtree(OUT / "augmented", ignore_errors=True)
augmented = tv.augment.augment_dataset(
    small, recipe, OUT / "augmented", copies=1, progress=False
)
augmented.summary()

## 3. Training and evaluation

`tv.train` works the same for every backend (`yolo`, `rfdetr`, `yolo_pose`).
This run starts from the COCO weights Ultralytics publishes and trains for
10 epochs on the demo frames, at 1280 pixels on the long side
because the players are small. It takes a few minutes on a Colab T4. The
score comes from `evaluate`, the same code for every backend.

In [ ]:
%%capture train_log
run = tv.train(
    "yolo",
    dataset,
    weights="yolo11n.pt",
    epochs=10,
    batch_size=8,
    imgsz=1280,
    rect=True,
    output_dir=OUT / "runs",
    name="detector",
    exist_ok=True,
)

In [ ]:
import pandas as pd

detector = run.load(conf=0.3)
pd.Series(detector.evaluate(dataset).as_dict()).round(3).to_frame("validation")

## 4. Inference

The pipeline detects the players, tracks them, maps them onto the pitch and
splits them into teams by kit. The camera does not move, so one homography
serves the whole clip: `fixed_homography` and `undistortion` take the
calibration that came with the data. (Broadcast footage has no such file;
there a pitch-landmark model passed as `keypoint_model` finds the homography
frame by frame, as in [04_inference](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/04_inference.ipynb).)

In [ ]:
clip = tv.data.load_labelled_clip(demo.clip_dir)
pipeline = tv.Pipeline(
    detector=detector,
    fixed_homography=demo.calibration,
    undistortion=demo.calibration,
    tracker="bytetrack",
    team_classifier=tv.teams.TeamClassifier(embedder="siglip"),
)
result = pipeline.run(clip.video_path, end=250, progress=False)  # 10 seconds
print(len(result), "frames,", len(result.track_ids), "tracks")
result.to_dataframe().query("object == 'person'").head()

The clip comes with the true boxes and identities, so the tracks can be
scored (MOTA and IDF1 of 1 would be a perfect run):

In [ ]:
tracking = tv.evaluation.evaluate_tracking(result, clip, hota=False)
pd.Series(tracking.as_dict()).dropna().round(3).to_frame("demo clip")

`export` writes the run folder (`result.pkl`, `tracks.csv`, StatsBomb-style
`freeze_frames.csv`, and the ball carrier and pass tables `carriers.csv` and
`passes.csv`, empty here because the demo has no ball), the same files
`tactifoot run` produces.

In [ ]:
run_folder = result.export(OUT / "run")
sorted(p.name for p in run_folder.iterdir())

## 5. Tactical map

One frame with tracks and teams, plus the top-down radar; then where each
team spent the clip.

In [ ]:
frame_result = result[200]
frame = tv.VideoReader(clip.video_path).read(frame_result.index)
# The pitch lines are left out: they are straight in the model and curved in a fisheye frame.
annotator = tv.viz.FrameAnnotator(style="video_game", draw_pitch_lines=False)
annotated = annotator.annotate(frame, frame_result)
radar = tv.viz.PitchRadar().draw(frame_result)
tv.show(
    tv.viz.overlay(annotated, radar, position="bottom-right", width_fraction=0.3),
    width=16,
)

In [ ]:
from matplotlib.figure import Figure

fig = Figure(figsize=(12, 4), layout="constrained")
for ax, team in zip(fig.subplots(1, 2), (0, 1), strict=True):
    tv.viz.plot_heatmap(result, team=team, ax=ax)
fig

## Next

* The module notebooks above go through every option.
* `tv.viz.render_video(result, clip.video_path, "annotated.mp4")` writes the
  annotated clip ([05_visualisation](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/05_visualisation.ipynb)).
* The command line runs the same steps on whole matches:
  `tactifoot init pipeline.yaml` writes a commented run file, then
  `tactifoot run pipeline.yaml --video match.mp4 --output-dir outputs/match`,
  which is `tv.run_file.load("pipeline.yaml").run("match.mp4", "outputs/match")`.

Video and annotations: SoccerTrack v2, A. Scott, I. Uchida, K. Kuroda, Y. Kim, K. Fujii, https://huggingface.co/datasets/atomscott/soccertrack-v2, licensed CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Changes: cut to a short excerpt, frames extracted, annotations converted, and model outputs drawn by tactifoot_vision.